In [1]:
# Carregar base de dados original
import pandas as pd

df = pd.read_csv('data/P1_Mestre_5epocas.csv')

In [2]:
# Selecionar apenas colunas essenciais
colunas_importantes = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HST', 'AST', 'B365H', 'B365D', 'B365A']
df_limpo = df[colunas_importantes].copy()

In [3]:
# Calcular pontos de cada jogo
df_limpo['Date'] = pd.to_datetime(df_limpo['Date'], format='mixed')
df_limpo = df_limpo.sort_values(by='Date')

def calcula_pontos_casa(resultado):
    if resultado == 'H': return 3
    elif resultado == 'D': return 1
    else: return 0

def calcula_pontos_fora(resultado):
    if resultado == 'A': return 3
    elif resultado == 'D': return 1
    else: return 0

df_limpo['HomePoints'] = df_limpo['FTR'].apply(calcula_pontos_casa)
df_limpo['AwayPoints'] = df_limpo['FTR'].apply(calcula_pontos_fora)

In [4]:
# Acumular pontos totais das equipas
df_limpo['HomeForm'] = 0
df_limpo['AwayForm'] = 0
pontos_totais = {}

for index, row in df_limpo.iterrows():
    casa = row['HomeTeam']
    fora = row['AwayTeam']
    
    if casa not in pontos_totais: pontos_totais[casa] = 0
    if fora not in pontos_totais: pontos_totais[fora] = 0
        
    df_limpo.at[index, 'HomeForm'] = pontos_totais[casa]
    df_limpo.at[index, 'AwayForm'] = pontos_totais[fora]
    
    pontos_totais[casa] += row['HomePoints']
    pontos_totais[fora] += row['AwayPoints']

In [5]:
# Calcular médias dos últimos jogos
N_JOGOS = 6
colunas_novas = ['HomeGolosMarcados_Ultimos6', 'HomeGolosSofridos_Ultimos6', 'HomePontos_Ultimos6', 'HomeRematesBaliza_Ultimos6', 
                 'AwayGolosMarcados_Ultimos6', 'AwayGolosSofridos_Ultimos6', 'AwayPontos_Ultimos6', 'AwayRematesBaliza_Ultimos6']

for col in colunas_novas:
    df_limpo[col] = 0.0

historico = {}
for index, row in df_limpo.iterrows():
    casa, fora = row['HomeTeam'], row['AwayTeam']
    
    if casa not in historico: historico[casa] = {'marcados': [], 'sofridos': [], 'pontos': [], 'remates': []}
    if fora not in historico: historico[fora] = {'marcados': [], 'sofridos': [], 'pontos': [], 'remates': []}
        
    if len(historico[casa]['pontos']) > 0:
        df_limpo.at[index, 'HomeGolosMarcados_Ultimos6'] = sum(historico[casa]['marcados']) / len(historico[casa]['marcados'])
        df_limpo.at[index, 'HomeGolosSofridos_Ultimos6'] = sum(historico[casa]['sofridos']) / len(historico[casa]['sofridos'])
        df_limpo.at[index, 'HomePontos_Ultimos6'] = sum(historico[casa]['pontos'])
        df_limpo.at[index, 'HomeRematesBaliza_Ultimos6'] = sum(historico[casa]['remates']) / len(historico[casa]['remates'])
        
    if len(historico[fora]['pontos']) > 0:
        df_limpo.at[index, 'AwayGolosMarcados_Ultimos6'] = sum(historico[fora]['marcados']) / len(historico[fora]['marcados'])
        df_limpo.at[index, 'AwayGolosSofridos_Ultimos6'] = sum(historico[fora]['sofridos']) / len(historico[fora]['sofridos'])
        df_limpo.at[index, 'AwayPontos_Ultimos6'] = sum(historico[fora]['pontos'])
        df_limpo.at[index, 'AwayRematesBaliza_Ultimos6'] = sum(historico[fora]['remates']) / len(historico[fora]['remates'])
        
    historico[casa]['marcados'].append(row['FTHG'])
    historico[casa]['sofridos'].append(row['FTAG'])
    historico[casa]['pontos'].append(row['HomePoints'])
    historico[casa]['remates'].append(row.get('HST', 0) if pd.notna(row.get('HST', 0)) else 0)
    
    historico[fora]['marcados'].append(row['FTAG'])
    historico[fora]['sofridos'].append(row['FTHG'])
    historico[fora]['pontos'].append(row['AwayPoints'])
    historico[fora]['remates'].append(row.get('AST', 0) if pd.notna(row.get('AST', 0)) else 0)
    
    for equipa in [casa, fora]:
        historico[equipa]['marcados'] = historico[equipa]['marcados'][-N_JOGOS:]
        historico[equipa]['sofridos'] = historico[equipa]['sofridos'][-N_JOGOS:]
        historico[equipa]['pontos']   = historico[equipa]['pontos'][-N_JOGOS:]
        historico[equipa]['remates']  = historico[equipa]['remates'][-N_JOGOS:]

In [6]:
# Calcular histórico de confrontos diretos
df_limpo['H2H_Pontos_Casa'] = 1.0 
h2h_memoria = {}

for index, row in df_limpo.iterrows():
    casa, fora = row['HomeTeam'], row['AwayTeam']
    confronto = tuple(sorted([casa, fora]))
    
    if confronto not in h2h_memoria: 
        h2h_memoria[confronto] = []
        
    if len(h2h_memoria[confronto]) > 0:
        pontos = sum([3 if v == casa else 1 if v == 'Empate' else 0 for v in h2h_memoria[confronto]])
        df_limpo.at[index, 'H2H_Pontos_Casa'] = pontos / len(h2h_memoria[confronto])
        
    vencedor_hoje = casa if row['FTR'] == 'H' else fora if row['FTR'] == 'A' else 'Empate'
    h2h_memoria[confronto].append(vencedor_hoje)
    h2h_memoria[confronto] = h2h_memoria[confronto][-6:]

In [7]:
# Contabilizar dias de descanso biológico
df_limpo['HomeDiasDescanso'] = 0
df_limpo['AwayDiasDescanso'] = 0
ultima_data = {}
LIMITE = 14 

for index, row in df_limpo.iterrows():
    casa, fora, data = row['HomeTeam'], row['AwayTeam'], row['Date']
    
    df_limpo.at[index, 'HomeDiasDescanso'] = min((data - ultima_data[casa]).days, LIMITE) if casa in ultima_data else LIMITE
    df_limpo.at[index, 'AwayDiasDescanso'] = min((data - ultima_data[fora]).days, LIMITE) if fora in ultima_data else LIMITE
        
    ultima_data[casa] = data
    ultima_data[fora] = data

In [8]:
# Medir moral e vitórias consecutivas
df_limpo['Home_Streak'] = 0
df_limpo['Away_Streak'] = 0
memoria_streak = {}

for index, row in df_limpo.iterrows():
    casa, fora, res = row['HomeTeam'], row['AwayTeam'], row['FTR']
    
    if casa not in memoria_streak: memoria_streak[casa] = 0
    if fora not in memoria_streak: memoria_streak[fora] = 0
        
    df_limpo.at[index, 'Home_Streak'] = memoria_streak[casa]
    df_limpo.at[index, 'Away_Streak'] = memoria_streak[fora]
    
    memoria_streak[casa] = (memoria_streak[casa] + 1 if memoria_streak[casa] > 0 else 1) if res == 'H' else (memoria_streak[casa] - 1 if memoria_streak[casa] < 0 else -1) if res == 'A' else 0
    memoria_streak[fora] = (memoria_streak[fora] + 1 if memoria_streak[fora] > 0 else 1) if res == 'A' else (memoria_streak[fora] - 1 if memoria_streak[fora] < 0 else -1) if res == 'H' else 0

In [9]:
# Atualizar ranking matemático do Elo
ELO_INICIAL = 1500.0
K_FACTOR = 30 
df_limpo['Home_Elo'] = 0.0
df_limpo['Away_Elo'] = 0.0
elo_equipas = {}

def calcular_elo(elo_a, elo_b, res_a):
    esperado_a = 1 / (1 + 10 ** ((elo_b - elo_a) / 400))
    return elo_a + K_FACTOR * (res_a - esperado_a)

for index, row in df_limpo.iterrows():
    casa, fora = row['HomeTeam'], row['AwayTeam']
    
    if casa not in elo_equipas: elo_equipas[casa] = ELO_INICIAL
    if fora not in elo_equipas: elo_equipas[fora] = ELO_INICIAL
        
    df_limpo.at[index, 'Home_Elo'] = elo_equipas[casa]
    df_limpo.at[index, 'Away_Elo'] = elo_equipas[fora]
    
    res_casa = 1.0 if row['FTR'] == 'H' else 0.0 if row['FTR'] == 'A' else 0.5
    res_fora = 1.0 - res_casa
        
    elo_equipas[casa] = calcular_elo(df_limpo.at[index, 'Home_Elo'], df_limpo.at[index, 'Away_Elo'], res_casa)
    elo_equipas[fora] = calcular_elo(df_limpo.at[index, 'Away_Elo'], df_limpo.at[index, 'Home_Elo'], res_fora)

In [10]:
# Atribuir pesos aos jogos (Decaimento Exponencial)
import numpy as np

df_limpo['Jogos_Casa_Realizados'] = df_limpo.groupby('HomeTeam').cumcount()

# 1. Encontrar a data do último jogo do dataset
data_mais_recente = df_limpo['Date'].max()

# 2. Calcular os dias passados desde cada jogo até à data mais recente
dias_passados = (data_mais_recente - df_limpo['Date']).dt.days

# 3. Aplicar o Decaimento Exponencial (Janela = 365 dias)
df_limpo['Peso_Jogo'] = np.exp(-dias_passados / 730)

# Opcional: Se quiseres manter a tua penalização extra aos arranques de época
df_limpo.loc[df_limpo['Jogos_Casa_Realizados'] < 6, 'Peso_Jogo'] *= 0.5

In [11]:
# Treinar e avaliar modelos finais (Teste Cronológico Real)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

pesos_totais = df_limpo['Peso_Jogo']
y = df_limpo['FTR']
# Mantemos as odds escondidas para evitar vício!
X = df_limpo.drop(columns=['Date', 'FTHG', 'FTAG', 'FTR', 'HomePoints', 'AwayPoints', 'HomeTeam', 'AwayTeam', 'Jogos_Casa_Realizados', 'Peso_Jogo', 'HST', 'AST', 'B365H', 'B365D', 'B365A'])

# A GRANDE MUDANÇA: adicionámos "shuffle=False" 
# O modelo vai treinar com os primeiros 80% do tempo e testar no futuro desconhecido (os últimos 20%)
X_train, X_test, y_train, y_test, pesos_train, pesos_test = train_test_split(X, y, pesos_totais, test_size=0.2, shuffle=False)

tradutor = LabelEncoder()
y_train_num = tradutor.fit_transform(y_train)
y_test_num = tradutor.transform(y_test)

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train, sample_weight=pesos_train)

xgb = XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=2, subsample=0.8, random_state=42)
xgb.fit(X_train, y_train_num, sample_weight=pesos_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=2, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [12]:
# --- NOVO MÓDULO: PREVISÃO DE GOLOS TOTAIS ---
from xgboost import XGBRegressor
import joblib

# 1. A nossa nova Variável Alvo é a SOMA dos golos (FTHG + FTAG)
y_golos = df_limpo['FTHG'] + df_limpo['FTAG']

# 2. Esconder as variáveis do futuro e as Odds (igual ao modelo principal)
X_golos = df_limpo.drop(columns=[
    'Date', 'FTHG', 'FTAG', 'FTR', 'HomePoints', 'AwayPoints', 
    'HomeTeam', 'AwayTeam', 'Jogos_Casa_Realizados', 'Peso_Jogo', 
    'HST', 'AST', 'B365H', 'B365D', 'B365A'
])

# 3. Treinar o Regressor de Golos
xgb_golos = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=2, random_state=42)
xgb_golos.fit(X_golos, y_golos)

# 4. Guardar este segundo "cérebro" no teu Mac
joblib.dump(xgb_golos, 'modelos/modelo_xgboost_golos.pkl')
print("✅ Cérebro de Golos treinado e exportado com sucesso!")

✅ Cérebro de Golos treinado e exportado com sucesso!


In [13]:
# --- NOVO MÓDULO: AMBAS MARCAM (BTTS) ---
from xgboost import XGBClassifier
import joblib

# 1. Criar a variável alvo: 1 se AMBOS marcarem, 0 se alguém ficar a zeros
y_btts = ((df_limpo['FTHG'] > 0) & (df_limpo['FTAG'] > 0)).astype(int)

# 2. Esconder variáveis do futuro e as Odds
X_btts = df_limpo.drop(columns=[
    'Date', 'FTHG', 'FTAG', 'FTR', 'HomePoints', 'AwayPoints', 
    'HomeTeam', 'AwayTeam', 'Jogos_Casa_Realizados', 'Peso_Jogo', 
    'HST', 'AST', 'B365H', 'B365D', 'B365A'
])

# 3. Treinar o Classificador do Ambas Marcam
xgb_btts = XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=2, random_state=42)
xgb_btts.fit(X_btts, y_btts)

# 4. Guardar este terceiro cérebro no teu Mac
joblib.dump(xgb_btts, 'modelos/modelo_xgboost_btts.pkl')
print("✅ Cérebro do Ambas Marcam (BTTS) treinado e exportado com sucesso!")

✅ Cérebro do Ambas Marcam (BTTS) treinado e exportado com sucesso!


In [14]:
from sklearn.metrics import accuracy_score
import math

print("--- 📊 RELATÓRIO DE ACCURACY (MERCADOS ALTERNATIVOS) ---")

# 1. Isolar os últimos 20% dos dados (o Teste Cronológico Real)
tamanho_teste = int(len(df_limpo) * 0.2)
indice_corte = len(df_limpo) - tamanho_teste

teste_df = df_limpo.iloc[indice_corte:].copy()
X_teste_geral = X_btts.iloc[indice_corte:] # As colunas limpas são iguais para todos os modelos

# --- TESTE 1: AMBAS MARCAM (BTTS) ---
# O que aconteceu na vida real?
y_teste_btts = ((teste_df['FTHG'] > 0) & (teste_df['FTAG'] > 0)).astype(int)
# O que o modelo previu?
preds_btts = xgb_btts.predict(X_teste_geral)
acc_btts = accuracy_score(y_teste_btts, preds_btts)
print(f"🏁 Accuracy Ambas Marcam (BTTS): {acc_btts * 100:.2f}%")

# --- TESTE 2: MAIS DE 2.5 GOLOS ---
# O que aconteceu na vida real?
y_teste_over25 = (teste_df['FTHG'] + teste_df['FTAG'] > 2).astype(int)

# O que o modelo previu (usando o Poisson)?
preds_golos_esperados = xgb_golos.predict(X_teste_geral)
preds_over25 = []

for lam in preds_golos_esperados:
    lam = max(0.1, float(lam))
    p0 = math.exp(-lam) * (lam**0) / math.factorial(0)
    p1 = math.exp(-lam) * (lam**1) / math.factorial(1)
    p2 = math.exp(-lam) * (lam**2) / math.factorial(2)
    prob_over25 = 1 - (p0 + p1 + p2)
    
    # Se a probabilidade for > 50%, o modelo assume que bate o +2.5
    if prob_over25 > 0.50:
        preds_over25.append(1)
    else:
        preds_over25.append(0)

acc_over25 = accuracy_score(y_teste_over25, preds_over25)
print(f"🥅 Accuracy +2.5 Golos: {acc_over25 * 100:.2f}%")

# --- TESTE 3: DUPLA HIPÓTESE (Confiança Alta) ---
# Na dupla hipótese, testar a accuracy normal é injusto porque o 1X é quase sempre favorito.
# Vamos ver qual é a taxa de acerto do modelo quando ele avisa que o 1X é "Muito Seguro" (>80%)
probs_1x2 = xgb.predict_proba(X_teste_geral)

acertos_1x = 0
apostas_1x = 0

for i in range(len(teste_df)):
    # XGBoost regressa a lista de probabilidades: 0=Fora(A), 1=Empate(D), 2=Casa(H)
    p_casa = probs_1x2[i][2]
    p_empate = probs_1x2[i][1]
    p_1x = p_casa + p_empate
    
    resultado_real = teste_df.iloc[i]['FTR']
    
    if p_1x >= 0.80:
        apostas_1x += 1
        if resultado_real in ['H', 'D']:
            acertos_1x += 1

if apostas_1x > 0:
    print(f"🛡️ Accuracy Dupla Hipótese (Quando a confiança no 1X é > 80%): {(acertos_1x/apostas_1x) * 100:.2f}% (Testado em {apostas_1x} jogos)")
else:
    print("🛡️ O modelo não encontrou jogos com confiança superior a 80% para testar.")

--- 📊 RELATÓRIO DE ACCURACY (MERCADOS ALTERNATIVOS) ---
🏁 Accuracy Ambas Marcam (BTTS): 67.76%
🥅 Accuracy +2.5 Golos: 60.53%
🛡️ Accuracy Dupla Hipótese (Quando a confiança no 1X é > 80%): 89.91% (Testado em 109 jogos)


In [15]:
# Simular lucros nas apostas desportivas
import pandas as pd
from sklearn.metrics import accuracy_score

# 1. Calcular e imprimir a Accuracy Geral do Modelo
acc_geral = accuracy_score(y_test_num, xgb.predict(X_test))
print(f"🎯 Accuracy Geral do XGBoost: {acc_geral * 100:.2f}%\n")

# 2. Preparar os dados para o Simulador
jogos_teste = df_limpo.loc[X_test.index, ['Date', 'HomeTeam', 'AwayTeam', 'FTR', 'B365H', 'B365D', 'B365A']].copy()
probabilidades = xgb.predict_proba(X_test)
jogos_teste[['Prob_Away', 'Prob_Draw', 'Prob_Home']] = probabilidades

resultados = []
for margem in [0.0, 0.05, 0.10, 0.15, 0.20]:
    banca = 1000.0
    apostas, ganhas = 0, 0
    
    for _, jogo in jogos_teste.iterrows():
        prob_h, prob_a = 1 / jogo['B365H'], 1 / jogo['B365A']
        vant_h, vant_a = jogo['Prob_Home'] - prob_h, jogo['Prob_Away'] - prob_a
        
        aposta, odd = (None, 0)
        if vant_h > margem and vant_h > vant_a: aposta, odd = ('H', jogo['B365H'])
        elif vant_a > margem: aposta, odd = ('A', jogo['B365A'])
            
        if aposta:
            apostas += 1
            banca -= 10.0
            if jogo['FTR'] == aposta:
                banca += (10.0 * odd)
                ganhas += 1
                
    lucro = banca - 1000.0
    # Calcular a Accuracy específica destas apostas
    taxa_acerto = (ganhas / apostas) * 100 if apostas > 0 else 0
    roi = (lucro / (apostas * 10.0)) * 100 if apostas > 0 else 0
    
    resultados.append({
        'Margem': f"{margem*100:.0f}%", 
        'Apostas': apostas, 
        'Accuracy (Apostas)': f"{taxa_acerto:.1f}%", 
        'Lucro': f"{lucro:.2f} €", 
        'ROI': f"{roi:.2f}%"
    })

print(pd.DataFrame(resultados).to_string(index=False))

🎯 Accuracy Geral do XGBoost: 58.36%

Margem  Apostas Accuracy (Apostas)     Lucro     ROI
    0%      177              28.2% -292.90 € -16.55%
    5%       66              25.8% -170.30 € -25.80%
   10%       19              42.1%   78.60 €  41.37%
   15%        7              57.1%   92.60 € 132.29%
   20%        1             100.0%   29.00 € 290.00%


In [16]:
# Guardar o cérebro, as colunas e os dados atuais para a Web App ler
import joblib
joblib.dump(xgb, 'modelos/modelo_xgboost.pkl')
joblib.dump(list(X.columns), 'modelos/colunas.pkl')
df_limpo.to_csv('data/base_processada.csv', index=False)